In [3]:
import pandas as pd
import numpy as np

preds = pd.read_csv("data/validation/analysis.csv")
game_info_df = pd.read_csv("s3://collegebasketballinsiders/box-scores/2026/game-info/game-info.csv")
map_df = pd.read_csv("s3://collegebasketballinsiders/general/map.csv", index_col=0)

In [4]:
game_info_df = game_info_df.merge(map_df[["espn_2", "team_id"]], left_on="home_team", right_on="espn_2")
game_info_df['home_team_id'] = game_info_df['team_id']
game_info_df = game_info_df[['game_id', 'date_utc', 'time_utc', 'neutral_site',
       'home_team', 'home_team_id', 'away_team', 'home_1h', 'away_1h', 'home_2h', 'away_2h',
       'home_score', 'away_score']]
game_info_df = game_info_df.merge(map_df[["espn_2", "team_id"]], left_on="away_team", right_on="espn_2")
game_info_df['away_team_id'] = game_info_df['team_id']
game_info_df = game_info_df[['game_id', 'date_utc', 'time_utc', 'neutral_site',
       'home_team', 'home_team_id', 'away_team', 'away_team_id', 'home_1h', 'away_1h', 'home_2h', 'away_2h',
       'home_score', 'away_score']]
game_info_df = game_info_df[~game_info_df['home_2h'].isna()]

In [7]:
game_info_df.columns

Index(['game_id', 'date_utc', 'time_utc', 'neutral_site', 'home_team',
       'home_team_id', 'away_team', 'away_team_id', 'home_1h', 'away_1h',
       'home_2h', 'away_2h', 'home_score', 'away_score'],
      dtype='object')

In [8]:
val_df = pd.merge(preds, game_info_df[["game_id", 'home_1h', 'away_1h',
       'home_2h', 'away_2h', 'home_score', 'away_score']], on="game_id")

In [11]:
import numpy as np
import pandas as pd

def _nz(x):  # pd.notna shortcut
    return pd.notna(x)

def _safe_logit(p, eps=1e-6):
    """
    Numerically stable logit. Clips p to [eps, 1-eps].
    """
    if pd.isna(p):
        return np.nan
    p = float(p)
    if p <= 0.0 or p >= 1.0:
        p = min(max(p, eps), 1.0 - eps)
    return np.log(p / (1.0 - p))

def _sigmoid(x):
    """Numerically safe logistic."""
    x = float(x)
    if x >= 0:
        z = np.exp(-x)
        return 1.0 / (1.0 + z)
    else:
        z = np.exp(x)
        return z / (1.0 + z)

def _symmetrize_home_win_prob(p_home, p_away):
    """
    Take home and away win probabilities (any subset) and
    return a single symmetric home win probability.

    Rules:
      - if both present: ph = p_home / (p_home + p_away)   (renormalize)
      - if only home present: ph = p_home
      - if only away present: ph = 1 - p_away
      - if neither: NaN
    """
    has_h = _nz(p_home)
    has_a = _nz(p_away)

    if has_h and has_a:
        denom = p_home + p_away
        if denom <= 0:
            return np.nan
        return float(p_home / denom)
    elif has_h:
        return float(p_home)
    elif has_a:
        return float(1.0 - p_away)
    else:
        return np.nan

def reconcile_game_all_preds(
    row,
    # Weights (set any to 0.0 to ignore that signal)
    w_full_score=1.0,        # pred_home_score, pred_away_score
    w_half_score=1.0,        # pred_home_1h, pred_away_1h, pred_home_2h, pred_away_2h
    w_totals=1.0,            # pred_total, pred_1h_total, pred_2h_total
    w_margin=1.0,            # pred_margin / pred_home_margin / pred_away_margin
    w_margin_1h=0.7,         # pred_1h_margin
    w_margin_2h=0.7,         # pred_2h_margin
    w_half_prior=0.05,       # soft prior: H1≈0.5*S_h, A1≈0.5*S_a
    use_both_game_margins=True,
    clip_nonneg=True,

    # Win-probability constraints
    w_winprob_full=0.8,      # weight for full-game win prob equation
    w_winprob_1h=0.6,        # weight for 1H win prob equation
    margin_scale_full=8.0,   # points per logit unit (full game)
    margin_scale_1h=5.0,     # points per logit unit (1H)
):
    """
    Solve for x = [S_h, S_a, H1, A1]^T using all available predictions.
    Uses:
      - score/total/margin models
      - calibrated win probabilities (full + 1H) as constraints on margins

    Expected win-prob columns (optional):
      - proba_home_win, proba_away_win
      - proba_home_1h_win, proba_away_1h_win

    Returns a Series with consensus scores/totals/margins plus:
      - cons_home_win_prob, cons_away_win_prob
      - cons_home_1h_win_prob, cons_away_1h_win_prob
    """

    # Unknowns: x = [S_h, S_a, H1, A1]
    A_rows, y_vals, w_vals = [], [], []

    # --- Fetch regression-style predictions (gracefully allow missing)
    Sh  = row.get('pred_home_score')
    Sa  = row.get('pred_away_score')
    H1  = row.get('pred_home_1h')
    A1p = row.get('pred_away_1h')
    H2  = row.get('pred_home_2h')
    A2  = row.get('pred_away_2h')

    T   = row.get('pred_total')        # game total
    T1  = row.get('pred_1h_total')     # 1H total
    T2  = row.get('pred_2h_total')     # 2H total

    Mg      = row.get('pred_margin')          # game margin (home - away)
    Mh      = row.get('pred_home_margin')     # home-side margin (should equal Mg)
    Ma      = row.get('pred_away_margin')     # away-side margin (use -Ma)
    M1      = row.get('pred_1h_margin')       # 1H margin
    M2      = row.get('pred_2h_margin')       # 2H margin

    # --- Fetch win probabilities (optional)
    # full game
    p_home_win   = row.get('proba_home_win')
    p_away_win   = row.get('proba_away_win')

    # 1H
    p_home_1h_win = row.get('proba_home_1h_win')
    p_away_1h_win = row.get('proba_away_1h_win')

    # =========================
    # Original linear equations
    # =========================

    # --- Full scores
    if _nz(Sh) and w_full_score > 0:
        A_rows.append([1.0, 0.0, 0.0, 0.0]); y_vals.append(Sh); w_vals.append(w_full_score)
    if _nz(Sa) and w_full_score > 0:
        A_rows.append([0.0, 1.0, 0.0, 0.0]); y_vals.append(Sa); w_vals.append(w_full_score)

    # --- 1H scores
    if _nz(H1) and w_half_score > 0:
        A_rows.append([0.0, 0.0, 1.0, 0.0]); y_vals.append(H1); w_vals.append(w_half_score)
    if _nz(A1p) and w_half_score > 0:
        A_rows.append([0.0, 0.0, 0.0, 1.0]); y_vals.append(A1p); w_vals.append(w_half_score)

    # --- 2H scores (S_h - H1) and (S_a - A1)
    if _nz(H2) and w_half_score > 0:
        A_rows.append([1.0, 0.0,-1.0, 0.0]); y_vals.append(H2); w_vals.append(w_half_score)
    if _nz(A2) and w_half_score > 0:
        A_rows.append([0.0, 1.0, 0.0,-1.0]); y_vals.append(A2); w_vals.append(w_half_score)

    # --- Totals
    if _nz(T) and w_totals > 0:
        A_rows.append([1.0, 1.0, 0.0, 0.0]); y_vals.append(T);  w_vals.append(w_totals)
    if _nz(T1) and w_totals > 0:
        A_rows.append([0.0, 0.0, 1.0, 1.0]); y_vals.append(T1); w_vals.append(w_totals)
    if _nz(T2) and w_totals > 0:
        A_rows.append([1.0, 1.0,-1.0,-1.0]); y_vals.append(T2); w_vals.append(w_totals)

    # --- Margins (scores)
    if _nz(Mg) and w_margin > 0:
        A_rows.append([1.0,-1.0,0.0,0.0]); y_vals.append(Mg); w_vals.append(w_margin)

    if use_both_game_margins and w_margin > 0:
        if _nz(Mh):
            A_rows.append([1.0,-1.0,0.0,0.0]); y_vals.append(Mh);  w_vals.append(0.5 * w_margin)
        if _nz(Ma):
            A_rows.append([1.0,-1.0,0.0,0.0]); y_vals.append(-Ma); w_vals.append(0.5 * w_margin)
    else:
        if (not _nz(Mg)) and w_margin > 0:
            if _nz(Mh):
                A_rows.append([1.0,-1.0,0.0,0.0]); y_vals.append(Mh);  w_vals.append(w_margin)
            elif _nz(Ma):
                A_rows.append([1.0,-1.0,0.0,0.0]); y_vals.append(-Ma); w_vals.append(w_margin)

    # 1H margin
    if _nz(M1) and w_margin_1h > 0:
        A_rows.append([0.0,0.0,1.0,-1.0]); y_vals.append(M1); w_vals.append(w_margin_1h)

    # 2H margin
    if _nz(M2) and w_margin_2h > 0:
        A_rows.append([1.0,-1.0,-1.0,1.0]); y_vals.append(M2); w_vals.append(w_margin_2h)

    # --- Soft priors: H1 ≈ 0.5*S_h, A1 ≈ 0.5*S_a
    if w_half_prior > 0:
        A_rows.append([-0.5, 0.0, 1.0, 0.0]); y_vals.append(0.0); w_vals.append(w_half_prior)
        A_rows.append([ 0.0,-0.5, 0.0, 1.0]); y_vals.append(0.0); w_vals.append(w_half_prior)

    # ======================================
    # Win-probability → implied margins
    # ======================================

    # Full-game win prob → implied full-game margin
    ph_full = _symmetrize_home_win_prob(p_home_win, p_away_win)
    if _nz(ph_full) and w_winprob_full > 0 and margin_scale_full > 0:
        m_full = margin_scale_full * _safe_logit(ph_full)
        if np.isfinite(m_full):
            # margin = S_h - S_a = [1, -1, 0, 0] · x
            A_rows.append([1.0, -1.0, 0.0, 0.0])
            y_vals.append(m_full)
            w_vals.append(w_winprob_full)

    # 1H win prob → implied 1H margin
    ph_1h = _symmetrize_home_win_prob(p_home_1h_win, p_away_1h_win)
    if _nz(ph_1h) and w_winprob_1h > 0 and margin_scale_1h > 0:
        m_1h = margin_scale_1h * _safe_logit(ph_1h)
        if np.isfinite(m_1h):
            # 1H margin = H1 - A1 = [0, 0, 1, -1] · x
            A_rows.append([0.0, 0.0, 1.0, -1.0])
            y_vals.append(m_1h)
            w_vals.append(w_winprob_1h)

    # If nothing to solve, return NaNs
    if not A_rows:
        return pd.Series({
            'cons_home_score':       np.nan,
            'cons_away_score':       np.nan,
            'cons_margin':           np.nan,
            'cons_home_1h':          np.nan,
            'cons_home_2h':          np.nan,
            'cons_away_1h':          np.nan,
            'cons_away_2h':          np.nan,
            'cons_total':            np.nan,
            'cons_1h_total':         np.nan,
            'cons_2h_total':         np.nan,
            'cons_home_win_prob':    np.nan,
            'cons_away_win_prob':    np.nan,
            'cons_home_1h_win_prob': np.nan,
            'cons_away_1h_win_prob': np.nan,
        })

    A = np.asarray(A_rows, dtype=float)  # (m,4)
    y = np.asarray(y_vals, dtype=float)  # (m,)
    w = np.asarray(w_vals, dtype=float)  # (m,)

    # Weighted least squares: argmin || W^{1/2}(Ax - y) ||^2
    Wsqrt = np.sqrt(w)[:, None]
    Aw = A * Wsqrt
    yw = y * Wsqrt.ravel()

    x_hat, *_ = np.linalg.lstsq(Aw, yw, rcond=None)
    S_h, S_a, H1_hat, A1_hat = x_hat.tolist()

    if clip_nonneg:
        S_h  = max(S_h, 0.0)
        S_a  = max(S_a, 0.0)
        H1_hat = min(max(H1_hat, 0.0), S_h)
        A1_hat = min(max(A1_hat, 0.0), S_a)

    H2_hat = S_h - H1_hat
    A2_hat = S_a - A1_hat
    M_hat  = S_h - S_a          # full-game margin
    M1_hat = H1_hat - A1_hat    # 1H margin

    # ======================================
    # Consensus probabilities from consensus margins
    # ======================================
    if margin_scale_full > 0:
        cons_home_win_prob = _sigmoid(M_hat / margin_scale_full)
        cons_away_win_prob = 1.0 - cons_home_win_prob
    else:
        cons_home_win_prob = np.nan
        cons_away_win_prob = np.nan

    if margin_scale_1h > 0:
        cons_home_1h_win_prob = _sigmoid(M1_hat / margin_scale_1h)
        cons_away_1h_win_prob = 1.0 - cons_home_1h_win_prob
    else:
        cons_home_1h_win_prob = np.nan
        cons_away_1h_win_prob = np.nan

    return pd.Series({
        'cons_home_score':       S_h,
        'cons_away_score':       S_a,
        'cons_margin':           M_hat,
        'cons_home_1h':          H1_hat,
        'cons_home_2h':          H2_hat,
        'cons_away_1h':          A1_hat,
        'cons_away_2h':          A2_hat,
        'cons_total':            S_h + S_a,
        'cons_1h_total':         H1_hat + A1_hat,
        'cons_2h_total':         H2_hat + A2_hat,
        'cons_home_win_prob':    cons_home_win_prob,
        'cons_away_win_prob':    cons_away_win_prob,
        'cons_home_1h_win_prob': cons_home_1h_win_prob,
        'cons_away_1h_win_prob': cons_away_1h_win_prob,
    })

def reconcile_dataframe_all_preds(
    df,
    w_full_score=1.0,
    w_half_score=1.0,
    w_totals=1.0,
    w_margin=1.0,
    w_margin_1h=0.7,
    w_margin_2h=0.7,
    w_half_prior=0.05,
    use_both_game_margins=True,
    clip_nonneg=True,

    # Win-prob weights & scales
    w_winprob_full=0.8,
    w_winprob_1h=0.6,
    margin_scale_full=8.0,
    margin_scale_1h=5.0,
):
    """
    Apply reconciliation to every row in a predictions DataFrame.

    Expected prediction column *names* (all optional; any subset is fine):
      - Scores: pred_home_score, pred_away_score
      - Halves: pred_home_1h, pred_home_2h, pred_away_1h, pred_away_2h
      - Totals: pred_total, pred_1h_total, pred_2h_total
      - Margins: pred_margin, pred_home_margin, pred_away_margin,
                 pred_1h_margin, pred_2h_margin
      - Win probs:
          * proba_home_win, proba_away_win
          * proba_home_1h_win, proba_away_1h_win

    Returns a copy with appended consensus columns:
      cons_home_score, cons_away_score, cons_margin,
      cons_home_1h, cons_home_2h, cons_away_1h, cons_away_2h,
      cons_total, cons_1h_total, cons_2h_total,
      cons_home_win_prob, cons_away_win_prob,
      cons_home_1h_win_prob, cons_away_1h_win_prob
    """
    df = df.copy()

    needed = [
        'pred_home_score','pred_away_score',
        'pred_home_1h','pred_home_2h','pred_away_1h','pred_away_2h',
        'pred_total','pred_1h_total','pred_2h_total',
        'pred_margin','pred_home_margin','pred_away_margin',
        'pred_1h_margin','pred_2h_margin',
        # win-prob columns
        'proba_home_win','proba_away_win',
        'proba_home_1h_win','proba_away_1h_win',
    ]
    for c in needed:
        if c not in df.columns:
            df[c] = np.nan

    out = df.apply(
        lambda r: reconcile_game_all_preds(
            r,
            w_full_score=w_full_score,
            w_half_score=w_half_score,
            w_totals=w_totals,
            w_margin=w_margin,
            w_margin_1h=w_margin_1h,
            w_margin_2h=w_margin_2h,
            w_half_prior=w_half_prior,
            use_both_game_margins=use_both_game_margins,
            clip_nonneg=clip_nonneg,
            w_winprob_full=w_winprob_full,
            w_winprob_1h=w_winprob_1h,
            margin_scale_full=margin_scale_full,
            margin_scale_1h=margin_scale_1h,
        ),
        axis=1
    )
    return pd.concat([df.reset_index(drop=True), out], axis=1)

In [12]:
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss, mean_absolute_error

def _build_base_val_df(val_df: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare a 'base' validation DF for tuning:
      - Drop any existing consensus columns (cons_*)
      - Keep predictions + truth columns
    """
    base = val_df.copy()

    # Drop any existing consensus columns so reconcile_dataframe_all_preds
    # doesn't create duplicate column names.
    cons_cols = [c for c in base.columns if c.startswith("cons_")]
    if cons_cols:
        base = base.drop(columns=cons_cols)

    return base


def _evaluate_reconciliation_params(
    base_val_df: pd.DataFrame,
    margin_scale_full: float,
    margin_scale_1h: float,
    w_winprob_full: float,
    w_winprob_1h: float,
    w_margin: float,
    w_margin_1h: float,
    # Fixed defaults for other weights (can be tuned later if desired)
    w_margin_2h: float = 0.7,
    w_full_score: float = 1.0,
    w_half_score: float = 1.0,
    w_totals: float = 1.0,
    w_half_prior: float = 0.05,
):
    """
    Run reconciliation with a given set of hyperparams and compute a scalar
    objective to MINIMIZE (lower is better).

    Objective combines:
      - full-game home win logloss
      - 1H home win logloss (ignoring tied halves)
      - margin MAE (scaled down to be on similar scale)
    """

    # Recompute consensus predictions with these hyperparameters
    rec = reconcile_dataframe_all_preds(
        base_val_df,
        w_full_score=w_full_score,
        w_half_score=w_half_score,
        w_totals=w_totals,
        w_margin=w_margin,
        w_margin_1h=w_margin_1h,
        w_margin_2h=w_margin_2h,
        w_half_prior=w_half_prior,
        w_winprob_full=w_winprob_full,
        w_winprob_1h=w_winprob_1h,
        margin_scale_full=margin_scale_full,
        margin_scale_1h=margin_scale_1h,
    )

    eps = 1e-6
    losses = []

    # -----------------------------
    # Full-game home win logloss
    # -----------------------------
    # True label: home_win = 1 if home_score > away_score
    mask_full = (
        rec["home_score"].notna()
        & rec["away_score"].notna()
        & rec["cons_home_win_prob"].notna()
    )
    if mask_full.any():
        y_full = (rec.loc[mask_full, "home_score"].values
                  > rec.loc[mask_full, "away_score"].values).astype(int)
        p_full = rec.loc[mask_full, "cons_home_win_prob"].clip(eps, 1 - eps).values
        loss_full = log_loss(y_full, p_full)
        losses.append(loss_full)
    else:
        loss_full = np.nan

    # -----------------------------
    # 1H home win logloss (ignore ties)
    # -----------------------------
    mask_1h = (
        rec["home_1h"].notna()
        & rec["away_1h"].notna()
        & (rec["home_1h"] != rec["away_1h"])  # ignore tied halves
        & rec["cons_home_1h_win_prob"].notna()
    )
    if mask_1h.any():
        y_1h = (rec.loc[mask_1h, "home_1h"].values
                > rec.loc[mask_1h, "away_1h"].values).astype(int)
        p_1h = rec.loc[mask_1h, "cons_home_1h_win_prob"].clip(eps, 1 - eps).values
        loss_1h = log_loss(y_1h, p_1h)
        losses.append(loss_1h)
    else:
        loss_1h = np.nan

    # -----------------------------
    # Final margin MAE
    # -----------------------------
    mask_marg = (
        rec["home_score"].notna()
        & rec["away_score"].notna()
        & rec["cons_margin"].notna()
    )
    if mask_marg.any():
        true_margin = (
            rec.loc[mask_marg, "home_score"].values
            - rec.loc[mask_marg, "away_score"].values
        )
        pred_margin = rec.loc[mask_marg, "cons_margin"].values
        mae_margin = mean_absolute_error(true_margin, pred_margin)
        # scale margin down so it's comparable to logloss
        margin_component = mae_margin / 10.0
        losses.append(margin_component)
    else:
        mae_margin = np.nan

    # -----------------------------
    # Aggregate objective
    # -----------------------------
    if losses:
        objective = float(np.mean(losses))
    else:
        objective = np.inf

    metrics = {
        "logloss_full": loss_full,
        "logloss_1h": loss_1h,
        "mae_margin": mae_margin,
        "objective": objective,
    }

    return objective, metrics


def tune_reconciliation_params(val_df: pd.DataFrame):
    """
    Main tuning function.

    val_df must contain:
      - prediction columns used by reconcile_dataframe_all_preds
      - truth columns: home_score, away_score, home_1h, away_1h

    Returns:
      best_params: dict of tuned hyperparameters
      best_metrics: dict of performance metrics for that setting
    """
    base_val_df = _build_base_val_df(val_df)

    # -----------------------------
    # Hyperparameter grids
    # (start small; you can expand later)
    # -----------------------------
    margin_scale_full_grid = [6.0, 8.0, 10.0]
    margin_scale_1h_grid   = [4.0, 5.0, 6.0]

    w_winprob_full_grid = [0.5, 0.8, 1.0]
    w_winprob_1h_grid   = [0.4, 0.6, 0.8]

    w_margin_grid    = [0.8, 1.0, 1.2]
    w_margin_1h_grid = [0.5, 0.7, 0.9]

    # Other weights left fixed at current defaults
    w_margin_2h = 0.7
    w_full_score = 1.0
    w_half_score = 1.0
    w_totals = 1.0
    w_half_prior = 0.05

    best_obj = np.inf
    best_params = None
    best_metrics = None

    for margin_scale_full in margin_scale_full_grid:
        for margin_scale_1h in margin_scale_1h_grid:
            for w_winprob_full in w_winprob_full_grid:
                for w_winprob_1h in w_winprob_1h_grid:
                    for w_margin in w_margin_grid:
                        for w_margin_1h in w_margin_1h_grid:
                            obj, metrics = _evaluate_reconciliation_params(
                                base_val_df=base_val_df,
                                margin_scale_full=margin_scale_full,
                                margin_scale_1h=margin_scale_1h,
                                w_winprob_full=w_winprob_full,
                                w_winprob_1h=w_winprob_1h,
                                w_margin=w_margin,
                                w_margin_1h=w_margin_1h,
                                w_margin_2h=w_margin_2h,
                                w_full_score=w_full_score,
                                w_half_score=w_half_score,
                                w_totals=w_totals,
                                w_half_prior=w_half_prior,
                            )

                            if obj < best_obj:
                                best_obj = obj
                                best_params = {
                                    "margin_scale_full": margin_scale_full,
                                    "margin_scale_1h": margin_scale_1h,
                                    "w_winprob_full": w_winprob_full,
                                    "w_winprob_1h": w_winprob_1h,
                                    "w_margin": w_margin,
                                    "w_margin_1h": w_margin_1h,
                                    "w_margin_2h": w_margin_2h,
                                    "w_full_score": w_full_score,
                                    "w_half_score": w_half_score,
                                    "w_totals": w_totals,
                                    "w_half_prior": w_half_prior,
                                }
                                best_metrics = metrics

    return best_params, best_metrics


# -----------------------------
# Example usage
# -----------------------------
if __name__ == "__main__":
    # assume you already have val_df loaded with the columns you listed
    # val_df = pd.read_parquet("your_validation_predictions_and_truth.parquet")

    best_params, best_metrics = tune_reconciliation_params(val_df)
    print("Best params:", best_params)
    print("Best metrics:", best_metrics)

    # Once you have best_params, you can use them for future daily preds:
    # daily_preds_with_consensus = reconcile_dataframe_all_preds(daily_preds, **best_params)


Best params: {'margin_scale_full': 6.0, 'margin_scale_1h': 6.0, 'w_winprob_full': 0.5, 'w_winprob_1h': 0.4, 'w_margin': 1.2, 'w_margin_1h': 0.9, 'w_margin_2h': 0.7, 'w_full_score': 1.0, 'w_half_score': 1.0, 'w_totals': 1.0, 'w_half_prior': 0.05}
Best metrics: {'logloss_full': 0.4786283444584726, 'logloss_1h': 0.5693454624477126, 'mae_margin': 12.506062638702224, 'objective': 0.7661933569254692}
